# Independent per-unit LightGBM with hydraulic LP features

This is an independent interpretation of another team's screenshot, not their implementation or a reproduction of their results. Fourteen separate binary gradient-boosted tree models predict disjoint unit columns. The per-unit predictors are not blended; each LightGBM predictor internally boosts multiple trees. Existing TCN and XGBoost sources are preserved.

The screenshot's exact 307/325/364/355/323 feature counts are not targets: its definitions, LP implementation and hyperparameters were unavailable. The actual feature inventory is generated from the matrices used here. Source-time storage and terminal S+168h values are used; projected storage comes only from the LP. Labels, Run No, optimizer runtime and observed future storage are not inputs.

## Experiments and interpretation

`base_topology` retains our previous 279-channel economics/water/topology design. `relative` adds local and intake/receiver relative-value descriptors using a shared within-case money scale. `unit_lp` adds nine plant schedules/margins and nine summaries of the unit's own LP schedule. `head_sensitivity` adds a single projected-head re-solve, +/-5% terminal-value re-solves, projected storage, soft-limit slack and extra own-plant summaries for multi-unit plants. `head_no_lio_relative` removes only the newly added relative columns for Lio. This is a tested hypothesis, not an assumption that Lio ignores economics.

The LP uses 28 six-hour blocks for computational cost and consistency with our retained TCN physics. Neural/tree targets and all output rows remain hourly over 168 hours. This resolution is inherited, not demonstrated optimal for LightGBM. Flow and minimum-volume constraints are block approximations. The LP omits startup costs, binary minimum loading, exact nonlinear losses and water travel times. Individual unit schedules can be degenerate among similar generators; they are features, not ground truth. Head updates are a one-pass approximation and need not improve the original LP objective. Terminal value conversion follows supplied generating routes/main-river flags with a 33-point initial-head best-point conversion inherited from our TCN; exact SHOP generation settings are unavailable.

## Why these training parameters

Binary logloss learns probabilities while the official selection metric is pooled micro ROC-AUC including cross-unit rankings. Learning rate .05, 31 leaves, minimum 100 rows per leaf and L2=10 are regularized starting heuristics, not established optimal values. Histogram max_bin=127 and four threads limit CPU/memory costs. Row/column subsampling .85/.9 regularizes the overlapping examples but does not make them independent. Deterministic CPU mode and seed42 allow reproducibility; they do not prove seed stability. No class reweighting or per-unit rank normalization is used because these can distort cross-unit rankings. LightGBM needs no learned normalization.

Development starts are split before hourly expansion, with a seven-day horizon purge. 2020/2021 scoring horizons stay inside their year. Per-unit logloss early stopping (60 rounds, cap800) uses those same development labels, so scores are tuning estimates rather than an untouched evaluation. A candidate is locked before exposed-2022 scoring at fixed development tree counts. All these years already have project exposure. The study is not full LOYO, and no public/private partition is assumed.

References: supplied `ML_Task_TDT4173.pdf`, `Dataset_definitions_and_explanation.pdf`, `kaggle_metric.py`; [LightGBM parameters](https://lightgbm.readthedocs.io/en/stable/Parameters.html) and [early stopping](https://lightgbm.readthedocs.io/en/stable/pythonapi/lightgbm.early_stopping.html). All data are supplied course resources. No external training data.

Run from the repository root. `check` performs no classifier fitting. `submit` requires an explicitly frozen configuration populated from measured results. Models fit raw data, never pretrained weights.


## Completed study

Measured results and limitations: `docs/lightgbm_hydraulic_20261101.md`. Frozen feature variant: `head_sensitivity`. Development mean micro AUC: 0.988025577. Exact tree budgets, feature schema hashes, raw hashes and dependency versions are embedded below; names are regenerated from the executed feature functions. Default check mode fits no classifier. Change only RUN_MODE for the selected recipe. The full raw-input export was fitted once in this study; repeat-fit or cross-machine bytewise reproducibility is not established. This is an experimental source, not official Short-notebook packaging.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from importlib import metadata
from time import perf_counter
import gc
import hashlib
import json
import re
import platform
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/kernel/Unit_commitment_decisions.csv").is_file()), Path.cwd())
RUN_MODE = "check"
FEATURE_VERSION = "hydro-decision-v1"
DECISION_GROUPS = {"economics", "water_balance"}
HOURS = 168
SEED = 42
THREADS = 4
DEADLINE_UTC = None
VARIANTS = ["base_topology", "relative", "unit_lp", "head_sensitivity", "head_no_lio_relative"]
MAX_ROUNDS = 800
STOPPING_ROUNDS = 60
PARAMETERS = dict(objective="binary", metric="binary_logloss", learning_rate=.05,
                  num_leaves=31, min_data_in_leaf=100, lambda_l2=10.,
                  feature_fraction=.9, bagging_fraction=.85, bagging_freq=1,
                  max_bin=127, num_threads=THREADS, deterministic=True,
                  force_col_wise=True, verbosity=-1)
SELECTED_CONFIGURATION = {'variant': 'head_sensitivity',
 'parameters': {'objective': 'binary',
                'metric': 'binary_logloss',
                'learning_rate': 0.05,
                'num_leaves': 31,
                'min_data_in_leaf': 100,
                'lambda_l2': 10.0,
                'feature_fraction': 0.9,
                'bagging_fraction': 0.85,
                'bagging_freq': 1,
                'max_bin': 127,
                'num_threads': 4,
                'deterministic': True,
                'force_col_wise': True,
                'verbosity': -1},
 'seed': 42,
 'iterations': {'Hogga_G1': 214,
                'Tokke_G1': 455,
                'Tokke_G2': 250,
                'Tokke_G3': 352,
                'Tokke_G4': 384,
                'Lio_G1': 228,
                'Byrte_G1': 272,
                'Vinje_G1': 279,
                'Vinje_G2': 200,
                'Vinje_G3': 316,
                'Haukeli_G1': 109,
                'Songa_G1': 228,
                'Kjela_G1': 377,
                'Vesle Kjela_G1': 232},
 'versions': {'lightgbm': '4.7.0',
              'numpy': '2.5.3',
              'pandas': '3.0.6',
              'scipy': '1.18.1',
              'scikit-learn': '1.9.1',
              'PyYAML': '6.0.3',
              'psutil': '7.2.2',
              'nbformat': '5.11.1'},
 'means': {'head_sensitivity': 0.98802557686607,
           'base_topology': 0.9702663320233506,
           'relative': 0.9803015852964179,
           'unit_lp': 0.9866701902001063,
           'head_no_lio_relative': 0.9878896180760388},
 'locked_utc': '2026-11-01T15:41:57.452231+00:00',
 'selection_uses_exposed_development': True,
 'stopping_uses_development_labels': True,
 'implementation_identity': '343527e7f9cebc6408f7ff140f55066e4f08e8649b33034ec86c10ef9301dfe3',
 'raw_sha256': {'labels': '6d51f8cc6869cb82b44708168808c305c1e007cf79878ca1c1d316d07aae3ab3',
                'mapping': 'c8b0e5bc500a5f1dec2337f5ac680847f6a7794a7c2b7a8fb19384910a029519',
                'template': '83e72b9e7299aff61065911af67437c81e8be2b0491ff012761d897c334f1cba',
                'prices': 'e5edd141b34a61139f3ee3022305175446d4b4f74618df85552c4f2b5e556ee4',
                'inflows': 'ff4f87d2768133b4a717c0ad366969244f7a86901695e1d4fef3c2ed8963ad66',
                'volumes': 'ef3d9c69699eef0d953b495ba69bf6676858e6b9c536087498135074912dd3b6',
                'water_values': 'a5f09ff96b551d3c297d90ccf3a0bbe92d8814b537ad63a46579cb0ff44aa4cb',
                'min_flow': '146062702f58920440fc064d461d544e9cd7c503c8df74ce96e721bee04629ea',
                'min_volume': '8b639b8339b84c5ff7db34ba4b4618e99d05a03819bf99fa5032a54490278806',
                'topology': '29cfa233cec7fc4b5401d0d571a566e6e17b7bb7b46a175abcd37a53fe175497'},
 'python_version': '3.14.7',
 'feature_schema_sha256': {'Hogga_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7',
                           'Tokke_G1': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Tokke_G2': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Tokke_G3': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Tokke_G4': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Lio_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7',
                           'Byrte_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7',
                           'Vinje_G1': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Vinje_G2': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Vinje_G3': 'cd72b8cd8407dd04c9fa4197c7613863dd71760b4b37e2125e713da49c6c37fa',
                           'Haukeli_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7',
                           'Songa_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7',
                           'Kjela_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7',
                           'Vesle Kjela_G1': 'e4e6cd6eddf28ae9c7f236de5bf6043eae99dc08921f38eb7dbf8dd5586516e7'}}
OUTPUT_ROOT = ROOT / "outputs/lightgbm_hydraulic"

FROZEN_CONFIGURATION_SHA256 = 'd9a18e094b6dfa7c661609b0d0b0ea85e9339e38a9d3f6b0decfda672bb803fb'


## Supplied raw inputs and exact target mapping

In [ ]:
RAW_FILES = {
    "labels": ROOT / "data/kernel/Unit_commitment_decisions.csv",
    "mapping": ROOT / "prediction_mapping.csv",
    "template": ROOT / "sample_submission.csv",
    "prices": ROOT / "data/kernel/Historical_day_ahead_price_2015_2025.csv",
    "inflows": ROOT / "data/kernel/Historical_inflow_1958_2025.csv",
    "volumes": ROOT / "data/kernel/Historical_volume_2015_2024.csv",
    "water_values": ROOT / "data/kernel/Synthetic_water_value_2015_2024.csv",
    "min_flow": ROOT / "data/extended/Constraint_min_flow.csv",
    "min_volume": ROOT / "data/extended/Constraint_min_volume.csv",
    "topology": ROOT / "data/extended/Tokke_Vinje_topology.yaml",
}
for path in RAW_FILES.values():
    if not path.is_file():
        raise FileNotFoundError(path)
train_cases = pd.read_csv(RAW_FILES["labels"])
test_cases = pd.read_csv(RAW_FILES["mapping"])
template = pd.read_csv(RAW_FILES["template"])
for cases in (train_cases, test_cases):
    cases["starttime"] = pd.to_datetime(cases["starttime"], utc=True, errors="raise")
    if cases.empty or cases["Run No"].isna().any() or not cases["Run No"].is_unique:
        raise ValueError("Cases require nonempty, unique, nonmissing Run No values.")
    if cases["starttime"].isna().any() or not cases["starttime"].is_unique:
        raise ValueError("Case start times must be unique and nonmissing.")
    if not cases["starttime"].equals(cases["starttime"].dt.normalize()):
        raise ValueError("Expected midnight UTC case starts.")
if not template["Run No"].is_unique or set(template["Run No"]) != set(test_cases["Run No"]):
    raise ValueError("Submission IDs must match the test mapping exactly.")
if set(train_cases["Run No"]) & set(test_cases["Run No"]):
    raise ValueError("Training and test case IDs overlap.")
train_cases = train_cases.sort_values("starttime").reset_index(drop=True)
test_cases = test_cases.set_index("Run No").loc[template["Run No"]].reset_index()
LABELS = template.columns.drop("Run No").tolist()
TARGETS = {}
for position, name in enumerate(LABELS):
    match = re.fullmatch(r"result_committed_(.+_G\d+)_t(\d+)", name)
    if match is None:
        raise ValueError(f"Unexpected target column: {name}")
    generator, hour = match.group(1), int(match.group(2))
    if hour in TARGETS.setdefault(generator, {}):
        raise ValueError(f"Duplicate generator/hour: {generator}, {hour}")
    TARGETS[generator][hour] = (name, position)
GENERATORS = list(TARGETS)
if len(GENERATORS) != 14 or len(LABELS) != 14 * HOURS:
    raise ValueError("Expected 14 generators and 2352 targets.")
for generator in GENERATORS:
    if set(TARGETS[generator]) != set(range(HOURS)):
        raise ValueError(f"Incomplete horizon for {generator}")
TARGET_COLUMNS = {g: [TARGETS[g][h][0] for h in range(HOURS)] for g in GENERATORS}
TARGET_POSITIONS = {g: [TARGETS[g][h][1] for h in range(HOURS)] for g in GENERATORS}
if not set(LABELS).issubset(train_cases) or not train_cases[LABELS].isin([0, 1]).all().all():
    raise ValueError("All target columns must be present with binary, nonmissing labels.")

def labels_for(cases, generator):
    return cases[TARGET_COLUMNS[generator]].to_numpy(dtype=np.uint8).reshape(-1)

all_starts = pd.DatetimeIndex(pd.concat([train_cases["starttime"], test_cases["starttime"]]))
FIRST_START = all_starts.min()
LAST_END = all_starts.max() + pd.Timedelta(hours=HOURS)
print(f"{len(train_cases)} training cases; {len(test_cases)} test cases; {len(LABELS)} targets")
display(pd.DataFrame({"generator": GENERATORS,
    "training_on_fraction": [float(labels_for(train_cases, g).mean()) for g in GENERATORS]}))

In [ ]:
def read_timeseries(key):
    frame = pd.read_csv(RAW_FILES[key])
    dates = pd.to_datetime(frame.pop("date"), utc=True, errors="raise")
    if dates.isna().any() or dates.duplicated().any():
        raise ValueError(f"Invalid or duplicate timestamps in {key}")
    frame.index = pd.DatetimeIndex(dates, name="date")
    frame = frame.sort_index().loc[FIRST_START:LAST_END]
    frame = frame.apply(pd.to_numeric, errors="raise").astype(np.float32)
    if frame.empty or not np.isfinite(frame.to_numpy()).all():
        raise ValueError(f"Empty or nonfinite input: {key}")
    return frame

tables = {key: read_timeseries(key) for key in
          ("prices", "inflows", "volumes", "water_values", "min_flow", "min_volume")}
if list(tables["prices"].columns) != ["Day-ahead price (EUR/MWh)"]:
    raise ValueError("Unexpected electricity price column.")
with RAW_FILES["topology"].open(encoding="utf-8") as stream:
    topology = yaml.safe_load(stream)
RESERVOIRS = list(tables["volumes"].columns)
if len(RESERVOIRS) != 17:
    raise ValueError("Expected 17 reservoirs.")
for key in ("inflows", "water_values"):
    if not set(RESERVOIRS).issubset(tables[key].columns):
        raise ValueError(f"Missing reservoir columns in {key}")
CAPACITY = {r: float(topology["model"]["reservoir"][r]["max_vol"]) for r in RESERVOIRS}
if any(v <= 0 for v in CAPACITY.values()):
    raise ValueError("Reservoir capacities must be positive.")
for key in ("inflows", "volumes", "min_flow", "min_volume"):
    if (tables[key].to_numpy() < 0).any():
        raise ValueError(f"Unexpected negative water quantity in {key}")

hourly_required = pd.date_range(FIRST_START, LAST_END, freq="h", inclusive="left")
for key in ("prices", "inflows", "min_flow", "min_volume"):
    used = tables[key].loc[tables[key].index < LAST_END]
    if not used.index.equals(hourly_required):
        raise ValueError(f"{key} must contain exactly one record per required hour.")
for key, timestamps in (("volumes", all_starts),
                        ("water_values", all_starts + pd.Timedelta(hours=HOURS))):
    if not timestamps.difference(tables[key].index).empty:
        raise ValueError(f"Missing required daily records in {key}")

connections = topology["connections"]
def supplying_reservoirs(name, kind, visited=None):
    visited = set() if visited is None else visited
    node = (kind, name)
    if node in visited:
        raise ValueError("Cycle in intake topology.")
    if kind == "reservoir":
        return {name}
    result = set()
    for edge in connections:
        if edge["to"] == name and edge["to_type"] == kind and edge["from_type"] in {"reservoir", "tunnel"}:
            result |= supplying_reservoirs(edge["from"], edge["from_type"], visited | {node})
    return result

FEEDERS = {}
for generator in GENERATORS:
    plants = [e["to"] for e in connections if e["from"] == generator
              and e["from_type"] == "generator" and e["to_type"] == "plant"]
    if len(plants) != 1:
        raise ValueError(f"Expected one plant for {generator}")
    FEEDERS[generator] = sorted(supplying_reservoirs(plants[0], "plant"))
    if not FEEDERS[generator] or not set(FEEDERS[generator]).issubset(RESERVOIRS):
        raise ValueError(f"Unknown supplying reservoirs for {generator}")
display(pd.DataFrame({"generator": GENERATORS,
                      "intake_reservoirs": [", ".join(FEEDERS[g]) for g in GENERATORS]}))
display(pd.DataFrame([{"input": k, "rows_in_task_range": len(v), "columns": len(v.columns),
                       "start": str(v.index.min()), "end": str(v.index.max())}
                      for k, v in tables.items()]))

# Traverse every supplied hydraulic predecessor, including river reaches and plants.
# A visited set prevents repeated counting at merges or cycles. These are possible
# upstream connections, not evidence that all upstream water arrives in this horizon.
def upstream_reservoirs(generator):
    plant = next(e["to"] for e in connections if e["from"] == generator
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    pending, visited, reservoirs = [("plant", plant)], set(), set()
    while pending:
        kind, name = pending.pop()
        if (kind, name) in visited:
            continue
        visited.add((kind, name))
        if kind == "reservoir":
            reservoirs.add(name)
        pending.extend((edge["from_type"], edge["from"]) for edge in connections
                       if edge["to_type"] == kind and edge["to"] == name
                       and edge["from_type"] in {"reservoir", "river", "tunnel", "plant"})
    return sorted(reservoirs)

UPSTREAM = {g: upstream_reservoirs(g) for g in GENERATORS}
OUTLET_ELEVATION = {}
for g in GENERATORS:
    plant = next(e["to"] for e in connections if e["from"] == g
                 and e["from_type"] == "generator" and e["to_type"] == "plant")
    OUTLET_ELEVATION[g] = float(topology["model"]["plant"][plant]["outlet_line"])
    if not set(FEEDERS[g]).issubset(UPSTREAM[g]) or not set(UPSTREAM[g]).issubset(RESERVOIRS):
        raise ValueError(f"Invalid upstream topology for {g}")
if FEATURE_VERSION == "hydro-context-v2":
    display(pd.DataFrame({"generator": GENERATORS,
                          "all_upstream_reservoirs": [", ".join(UPSTREAM[g]) for g in GENERATORS]}))

# Directed hydraulic edges exclude generator membership edges. Stop at the first
# receiving reservoir; never interpret graph connectivity as realized routing.
HYDRAULIC_KINDS = {"reservoir", "plant", "river", "tunnel"}
FORWARD, BACKWARD = {}, {}
for edge in connections:
    if edge["from_type"] in HYDRAULIC_KINDS and edge["to_type"] in HYDRAULIC_KINDS:
        left = (edge["from_type"], edge["from"])
        right = (edge["to_type"], edge["to"])
        FORWARD.setdefault(left, set()).add(right)
        BACKWARD.setdefault(right, set()).add(left)
PLANT_FOR = {g: next(e["to"] for e in connections if e["from"] == g
                     and e["from_type"] == "generator" and e["to_type"] == "plant")
             for g in GENERATORS}
PLANT_GENERATORS = {p: [g for g in GENERATORS if PLANT_FOR[g] == p]
                    for p in topology["model"]["plant"]}
PLANT_FEEDERS = {p: FEEDERS[gens[0]] for p, gens in PLANT_GENERATORS.items()}

def reachable(start, adjacency, stop_at_reservoir=False):
    pending, seen = list(adjacency.get(start, [])), set()
    while pending:
        node = pending.pop()
        if node == start or node in seen:
            continue
        seen.add(node)
        if not (stop_at_reservoir and node[0] == "reservoir"):
            pending.extend(adjacency.get(node, []))
    return seen

RECEIVERS = {p: sorted(name for kind, name in reachable(("plant", p), FORWARD, True)
                       if kind == "reservoir") for p in PLANT_GENERATORS}
UPSTREAM_PLANTS = {p: sorted(name for kind, name in reachable(("plant", p), BACKWARD)
                             if kind == "plant") for p in PLANT_GENERATORS}
DOWNSTREAM_PLANTS = {p: sorted(name for kind, name in reachable(("plant", p), FORWARD)
                               if kind == "plant") for p in PLANT_GENERATORS}
LEVEL_CURVES = {r: (np.asarray(topology["model"]["reservoir"][r]["vol_head"]["x"], dtype=float),
                    np.asarray(topology["model"]["reservoir"][r]["vol_head"]["y"], dtype=float))
                for r in RESERVOIRS}
for r, (x, y) in LEVEL_CURVES.items():
    if len(x) != len(y) or not np.all(np.diff(x) > 0) or not np.all(np.diff(y) >= 0):
        raise ValueError(f"Invalid volume-level curve: {r}")
GEN_SPECS = topology["model"]["generator"]
display(pd.DataFrame([{"plant": p, "intakes": PLANT_FEEDERS[p], "receivers": RECEIVERS[p],
                       "upstream_plants": UPSTREAM_PLANTS[p], "downstream_plants": DOWNSTREAM_PLANTS[p]}
                      for p in PLANT_GENERATORS]))

## Base economic and named topology descriptors

In [ ]:
def economic_context(margin, price):
    """Whole-horizon price/value descriptors, not a dispatch profitability solver."""
    margin, price = np.asarray(margin, dtype=np.float64), np.asarray(price, dtype=np.float64)
    n, width = margin.shape
    if width != HOURS or price.shape != margin.shape:
        raise ValueError("Economic context requires one complete horizon per row.")
    positive = margin > 0
    positive_margin = np.maximum(margin, 0)
    left_length = np.zeros_like(margin)
    right_length = np.zeros_like(margin)
    left_sum = np.zeros_like(margin)
    right_sum = np.zeros_like(margin)
    next_positive = np.full_like(margin, HOURS)
    previous_positive = np.full_like(margin, HOURS)
    # Each recurrence resets independently for every scheduling case.
    for h in range(width):
        left_length[:, h] = np.where(positive[:, h], (left_length[:, h-1] if h else 0) + 1, 0)
        left_sum[:, h] = np.where(positive[:, h], (left_sum[:, h-1] if h else 0) + margin[:, h], 0)
        previous_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((previous_positive[:, h-1] if h else HOURS) + 1, HOURS))
    for h in range(width - 1, -1, -1):
        right_length[:, h] = np.where(positive[:, h], (right_length[:, h+1] if h < width-1 else 0) + 1, 0)
        right_sum[:, h] = np.where(positive[:, h], (right_sum[:, h+1] if h < width-1 else 0) + margin[:, h], 0)
        next_positive[:, h] = np.where(positive[:, h], 0,
            np.minimum((next_positive[:, h+1] if h < width-1 else HOURS) + 1, HOURS))
    suffix_count = np.arange(width, 0, -1)
    suffix_positive = positive[:, ::-1].cumsum(1)[:, ::-1]
    suffix_margin = positive_margin[:, ::-1].cumsum(1)[:, ::-1]
    result = {
        "economic_margin_over_week_price_scale": margin / (price.std(1, keepdims=True) + 1.0),
        "economic_positive_streak_elapsed_h": left_length,
        "economic_positive_streak_remaining_h": right_length,
        "economic_positive_streak_total_h": left_length + right_length - positive,
        "economic_positive_streak_margin_sum": left_sum + right_sum - positive_margin,
        "economic_hours_to_next_positive": next_positive,
        "economic_hours_since_previous_positive": previous_positive,
        "economic_positive_fraction_remaining": suffix_positive / suffix_count,
        "economic_positive_margin_remaining_mean": suffix_margin / suffix_count,
        "economic_better_remaining_margin": np.maximum.accumulate(margin[:, ::-1], axis=1)[:, ::-1] - margin,
    }
    indices = np.arange(width)
    for window in (6, 24):
        end = np.minimum(indices + window, width)
        start = np.maximum(indices - window + 1, 0)
        prefix = np.concatenate([np.zeros((n, 1)), margin.cumsum(1)], axis=1)
        result[f"economic_next_{window}h_margin_mean"] = (prefix[:, end] - prefix[:, indices]) / (end - indices)
        result[f"economic_previous_{window}h_margin_mean"] = (prefix[:, indices + 1] - prefix[:, start]) / (indices + 1 - start)
    return result


def water_context(initial, cumulative, remaining, limits, capacities):
    """Local natural-inflow scenarios with zero releases/transfers, not future storage."""
    initial = np.asarray(initial, dtype=np.float64)
    cumulative = np.asarray(cumulative, dtype=np.float64)
    remaining = np.asarray(remaining, dtype=np.float64)
    limits = np.asarray(limits, dtype=np.float64)
    capacities = np.asarray(capacities, dtype=np.float64)
    if cumulative.shape != remaining.shape or limits.shape != cumulative.shape or             cumulative.shape[1:] != (HOURS, len(capacities)) or initial.shape != (len(cumulative), len(capacities)):
        raise ValueError("Invalid water-context dimensions.")
    if (capacities <= 0).any():
        raise ValueError("Reservoir capacities must be positive.")
    capacity = capacities.sum()
    start = initial[:, None, :]
    no_release = start + cumulative
    end_no_release = no_release + remaining
    headroom = np.maximum(capacities - start, 0)
    # A 1% capacity floor bounds the ratio near a full reservoir; apply per reservoir
    # before aggregation so spare space elsewhere does not cancel local pressure.
    pressure = np.log1p((cumulative + remaining) / np.maximum(headroom, 0.01 * capacities))
    future_floor = np.maximum.accumulate(limits[:, ::-1, :], axis=1)[:, ::-1, :]
    return {
        "water_week_inflow_to_headroom_log_max": pressure.max(2),
        "water_week_inflow_to_headroom_log_weighted": (pressure * capacities).sum(2) / capacity,
        "water_no_release_excess_fraction": np.maximum(no_release - capacities, 0).sum(2) / capacity,
        "water_end_no_release_excess_fraction": np.maximum(end_no_release - capacities, 0).sum(2) / capacity,
        "water_no_release_excess_max_fraction": np.maximum((no_release - capacities) / capacities, 0).max(2),
        "water_initial_margin_to_floor_fraction": (start - limits).sum(2) / capacity,
        "water_initial_margin_to_future_floor_fraction": (start - future_floor).sum(2) / capacity,
        "water_no_release_floor_deficit_fraction": np.maximum(limits - no_release, 0).sum(2) / capacity,
        "water_no_release_min_margin_fraction": ((no_release - limits) / capacities).min(2),
    }


def lookup(key, timestamps, columns=None):
    frame = tables[key] if columns is None else tables[key][columns]
    selected = frame.reindex(pd.DatetimeIndex(timestamps))
    if selected.isna().any().any():
        raise ValueError(f"Missing values while looking up {key}")
    return selected.to_numpy(dtype=np.float32)

def feature_matrix(cases):
    starts = pd.DatetimeIndex(cases["starttime"])
    if len(starts) == 0:
        raise ValueError("Cannot construct features for zero cases.")
    n = len(starts)
    # Use datetime arithmetic rather than assuming pandas timestamps use nanoseconds.
    timestamps = pd.DatetimeIndex(np.repeat(starts.to_numpy(), HOURS)) + pd.to_timedelta(
        np.tile(np.arange(HOURS), n), unit="h")
    price = lookup("prices", timestamps).reshape(n, HOURS)
    initial = lookup("volumes", starts, RESERVOIRS)
    terminal = lookup("water_values", starts + pd.Timedelta(hours=HOURS), RESERVOIRS)
    inflows = lookup("inflows", timestamps).reshape(n, HOURS, -1)
    features = {}

    def add(name, values):
        values = np.asarray(values, dtype=np.float32)
        if values.shape == (n,):
            values = np.repeat(values, HOURS)
        elif values.shape == (n, HOURS):
            values = values.reshape(-1)
        if values.shape != (n * HOURS,) or not np.isfinite(values).all():
            raise ValueError(f"Invalid feature {name}: {values.shape}")
        features[name] = values

    h = np.tile(np.arange(HOURS, dtype=np.float32), n)
    add("horizon_hour", h)
    add("hours_remaining", HOURS - 1 - h)
    add("hour_of_day", timestamps.hour.to_numpy())
    add("day_of_week", timestamps.dayofweek.to_numpy())
    add("month", timestamps.month.to_numpy())
    add("day_of_year_sin", np.sin(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("day_of_year_cos", np.cos(2 * np.pi * timestamps.dayofyear.to_numpy() / 365.25))
    add("price", price)
    for stat, values in (("mean", price.mean(1)), ("std", price.std(1)),
                         ("min", price.min(1)), ("max", price.max(1))):
        add(f"price_week_{stat}", values)
    add("price_minus_week_mean", price - price.mean(1, keepdims=True))
    add("price_week_rank", pd.DataFrame(price).rank(axis=1, pct=True).to_numpy())
    daily = price.reshape(n, 7, 24)
    for stat, values in (("mean", daily.mean(2)), ("min", daily.min(2)), ("max", daily.max(2))):
        add(f"price_day_{stat}", np.repeat(values, 24, axis=1))
    add("price_prefix_mean", price.cumsum(1) / np.arange(1, HOURS + 1))
    add("price_suffix_mean", price[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1))
    # Clip at the case boundary, not at the global dataset boundary.
    for offset in (-24, -6, -1, 1, 6, 24):
        add(f"price_offset_{offset:+d}", price[:, np.clip(np.arange(HOURS) + offset, 0, HOURS - 1)])
    for j, reservoir in enumerate(RESERVOIRS):
        add(f"initial_volume__{reservoir}", initial[:, j])
        add(f"initial_fraction__{reservoir}", initial[:, j] / CAPACITY[reservoir])
        add(f"terminal_value__{reservoir}", terminal[:, j])
        add(f"price_minus_value__{reservoir}", price - terminal[:, j, None])
    for j, name in enumerate(tables["inflows"].columns):
        flow = inflows[:, :, j]
        cumulative = flow.cumsum(1) * (3600 / 1e6)
        add(f"inflow__{name}", flow)
        add(f"inflow_week_mean__{name}", flow.mean(1))
        add(f"inflow_cumulative_Mm3__{name}", cumulative)
        add(f"inflow_remaining_Mm3__{name}", cumulative[:, -1:] - cumulative)
    for key in ("min_flow", "min_volume"):
        limits = lookup(key, timestamps).reshape(n, HOURS, -1)
        for j, name in enumerate(tables[key].columns):
            add(f"{key}__{name}", limits[:, :, j])
            add(f"{key}_week_max__{name}", limits[:, :, j].max(1))
            if key == "min_volume":
                add(f"initial_margin_to_min__{name}", initial[:, RESERVOIRS.index(name), None] - limits[:, :, j])
    if FEATURE_VERSION == "hydro-context-v2":
        # Rolling summaries stay strictly inside each supplied planning horizon.
        add("price_past_24h_mean", pd.DataFrame(price.T).rolling(24, min_periods=1).mean().to_numpy().T)
        add("price_next_24h_mean", pd.DataFrame(price[:, ::-1].T).rolling(24, min_periods=1).mean().to_numpy().T[:, ::-1])
        add("price_suffix_max", np.maximum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        add("price_suffix_min", np.minimum.accumulate(price[:, ::-1], axis=1)[:, ::-1])
        for j, reservoir in enumerate(RESERVOIRS):
            curve = topology["model"]["reservoir"][reservoir]["vol_head"]
            curve_x, curve_y = np.asarray(curve["x"]), np.asarray(curve["y"])
            if len(curve_x) != len(curve_y) or not np.all(np.diff(curve_x) > 0):
                raise ValueError(f"Invalid volume-level curve for {reservoir}")
            # np.interp clips outside the supplied curve: a static descriptor only.
            add(f"initial_level_m__{reservoir}", np.interp(initial[:, j], curve_x, curve_y))
    return pd.DataFrame(features, dtype=np.float32)

def generator_features(base, generator):
    # Copy only one generator's matrix at a time; never materialize all 14 together.
    X = base.copy()
    feeders = FEEDERS[generator]
    capacity = sum(CAPACITY[r] for r in feeders)
    total_initial = sum(base[f"initial_volume__{r}"] for r in feeders)
    value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in feeders) / capacity
    cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in feeders)
    remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in feeders)
    X["local_initial_fraction"] = total_initial / capacity
    X["local_terminal_value_capacity_weighted"] = value
    X["local_price_minus_terminal_value"] = base["price"] - value
    X["local_inflow"] = sum(base[f"inflow__{r}"] for r in feeders)
    X["local_cumulative_inflow_fraction"] = cumulative / capacity
    X["local_remaining_inflow_fraction"] = remaining / capacity
    X["local_available_water_proxy_fraction"] = (total_initial + cumulative + remaining) / capacity
    if FEATURE_VERSION == "hydro-context-v2":
        upstream = UPSTREAM[generator]
        upstream_capacity = sum(CAPACITY[r] for r in upstream)
        upstream_initial = sum(base[f"initial_volume__{r}"] for r in upstream)
        upstream_cumulative = sum(base[f"inflow_cumulative_Mm3__{r}"] for r in upstream)
        upstream_remaining = sum(base[f"inflow_remaining_Mm3__{r}"] for r in upstream)
        upstream_value = sum(base[f"terminal_value__{r}"] * CAPACITY[r] for r in upstream) / upstream_capacity
        X["upstream_initial_fraction"] = upstream_initial / upstream_capacity
        X["upstream_terminal_value_capacity_weighted"] = upstream_value
        X["upstream_price_minus_value"] = base["price"] - upstream_value
        X["upstream_inflow"] = sum(base[f"inflow__{r}"] for r in upstream)
        X["upstream_cumulative_inflow_fraction"] = upstream_cumulative / upstream_capacity
        X["upstream_remaining_inflow_fraction"] = upstream_remaining / upstream_capacity
        X["upstream_available_water_proxy_fraction"] = (upstream_initial + upstream_cumulative + upstream_remaining) / upstream_capacity
        initial_level = sum(base[f"initial_level_m__{r}"] * CAPACITY[r] for r in feeders) / capacity
        X["local_initial_gross_head_proxy_m"] = initial_level - OUTLET_ELEVATION[generator]
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        positive = (margin > 0).astype(np.float32)
        X["local_hours_above_value_remaining"] = positive[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
        X["local_positive_margin_remaining"] = np.maximum(margin, 0)[:, ::-1].cumsum(1)[:, ::-1].reshape(-1)
    extra = {}
    if "economics" in DECISION_GROUPS:
        margin = (base["price"] - value).to_numpy().reshape(-1, HOURS)
        extra.update(economic_context(margin, base["price"].to_numpy().reshape(-1, HOURS)))
    if "water_balance" in DECISION_GROUPS:
        def cube(prefix):
            return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS) for r in feeders], axis=2)
        initial = cube("initial_volume")[:, 0, :]
        cumulative_cube = cube("inflow_cumulative_Mm3")
        remaining_cube = cube("inflow_remaining_Mm3")
        floor = np.stack([base[f"min_volume__{r}"].to_numpy().reshape(-1, HOURS)
                          if f"min_volume__{r}" in base else np.zeros_like(cumulative_cube[:, :, 0])
                          for r in feeders], axis=2)
        extra.update(water_context(initial, cumulative_cube, remaining_cube, floor,
                                   [CAPACITY[r] for r in feeders]))
    if extra:
        X = pd.concat([X, pd.DataFrame({name: values.reshape(-1).astype(np.float32)
                                       for name, values in extra.items()}, index=X.index)], axis=1)
    if not np.isfinite(X.to_numpy()).all():
        raise ValueError("Features must be finite.")
    return X.astype(np.float32, copy=False)
reference_feature_matrix = feature_matrix
reference_generator_features = generator_features

def feature_matrix(cases):
    frame = reference_feature_matrix(cases)
    # Internal alignment metadata never enters the numeric feature matrix.
    frame.attrs["case_starts"] = pd.DatetimeIndex(cases["starttime"])
    return frame

def state(base, reservoirs, prefix):
    return np.stack([base[f"{prefix}__{r}"].to_numpy().reshape(-1, HOURS)
                     for r in reservoirs], axis=2)

def plant_state(base, plant):
    feeders = PLANT_FEEDERS[plant]
    weights = np.array([CAPACITY[r] for r in feeders], dtype=np.float64)
    volumes = state(base, feeders, "initial_volume")
    values = state(base, feeders, "terminal_value")
    level = np.stack([np.interp(volumes[:, :, i], *LEVEL_CURVES[r]) for i, r in enumerate(feeders)], axis=2)
    intake_level = np.average(level, weights=weights, axis=2)
    receiver = RECEIVERS[plant]
    outlet = float(topology["model"]["plant"][plant]["outlet_line"])
    tailwater = np.full_like(intake_level, outlet)
    if receiver:
        receiver_levels = np.stack([np.interp(base[f"initial_volume__{r}"].to_numpy().reshape(-1, HOURS),
                                              *LEVEL_CURVES[r]) for r in receiver], axis=2)
        tailwater = np.maximum(tailwater, receiver_levels.max(axis=2))
    return {"head": np.maximum(intake_level-tailwater, 1.0),
            "intake_level": intake_level, "tailwater": tailwater,
            "fraction": volumes.sum(axis=2)/weights.sum(),
            "value": np.average(values, weights=weights, axis=2)}

def topology_features(base, generator):
    plant = PLANT_FOR[generator]
    feeders, upstream, receiver = FEEDERS[generator], UPSTREAM[generator], RECEIVERS[plant]
    local = plant_state(base, plant)
    price = base["price"].to_numpy().reshape(-1, HOURS)
    capacity = np.array([CAPACITY[r] for r in upstream])
    volumes = state(base, upstream, "initial_volume")
    fractions = volumes / capacity
    terminal = state(base, upstream, "terminal_value")
    cumulative = state(base, upstream, "inflow_cumulative_Mm3")
    remaining = state(base, upstream, "inflow_remaining_Mm3")
    root_capacity = capacity.sum()
    pressure = (cumulative+remaining)/np.maximum(capacity-volumes, 0.01*capacity)
    extra = {
        "topo_initial_intake_level_m": local["intake_level"],
        "topo_initial_tailwater_m": local["tailwater"],
        "topo_initial_gross_head_m": local["head"],
        "topo_head_relative_reference": local["head"]/np.mean([c["ref"] for c in GEN_SPECS[generator]["turb_eff_curves"]]),
        "topo_upstream_initial_fraction": volumes.sum(2)/root_capacity,
        "topo_upstream_min_fraction": fractions.min(2),
        "topo_upstream_max_fraction": fractions.max(2),
        "topo_upstream_std_fraction": fractions.std(2),
        "topo_upstream_value": np.average(terminal, weights=capacity, axis=2),
        "topo_upstream_value_min": terminal.min(2),
        "topo_upstream_value_max": terminal.max(2),
        "topo_upstream_value_std": terminal.std(2),
        "topo_upstream_inflow": state(base, upstream, "inflow").sum(2),
        "topo_upstream_cumulative_fraction": cumulative.sum(2)/root_capacity,
        "topo_upstream_remaining_fraction": remaining.sum(2)/root_capacity,
        "topo_upstream_water_pressure_max": np.log1p(pressure).max(2),
        "topo_upstream_excess_fraction": np.maximum(volumes+cumulative+remaining-capacity, 0).sum(2)/root_capacity,
        "topo_upstream_price_minus_value": price-np.average(terminal, weights=capacity, axis=2),
    }
    for label, reservoirs in [("receiver", receiver), ("remote_upstream", sorted(set(upstream)-set(feeders)))]:
        if reservoirs:
            cap = np.array([CAPACITY[r] for r in reservoirs])
            vol = state(base, reservoirs, "initial_volume")
            wv = np.average(state(base, reservoirs, "terminal_value"), weights=cap, axis=2)
            frac = vol.sum(2)/cap.sum()
            inflow = state(base, reservoirs, "inflow").sum(2)
            free = np.maximum(cap-vol,0).sum(2)/cap.sum()
        else:
            frac = wv = inflow = free = np.zeros_like(price)
        extra.update({f"topo_{label}_fraction": frac, f"topo_{label}_value": wv,
                      f"topo_{label}_inflow": inflow, f"topo_{label}_free_fraction": free,
                      f"topo_local_minus_{label}_value": local["value"]-wv})
    for direction, related in [("upstream", UPSTREAM_PLANTS[plant]), ("downstream", DOWNSTREAM_PLANTS[plant])]:
        related_states = [plant_state(base,p) for p in related]
        margins = np.stack([price-s["value"] for s in related_states],axis=2) if related_states else np.zeros((*price.shape,1))
        fills = np.stack([s["fraction"] for s in related_states],axis=2) if related_states else np.zeros((*price.shape,1))
        extra.update({f"topo_{direction}_plant_margin_mean": margins.mean(2),
                      f"topo_{direction}_plant_margin_min": margins.min(2),
                      f"topo_{direction}_plant_margin_max": margins.max(2),
                      f"topo_{direction}_plant_positive_fraction": (margins>0).mean(2),
                      f"topo_{direction}_plant_fill_mean": fills.mean(2),
                      f"topo_{direction}_plant_fill_min": fills.min(2),
                      f"topo_{direction}_plant_fill_max": fills.max(2)})
    for r in RESERVOIRS:
        extra[f"topo_initial_level_m__{r}"] = np.interp(base[f"initial_volume__{r}"].to_numpy().reshape(-1,HOURS),*LEVEL_CURVES[r])
    # The source of each min-flow constraint is explicit in the hydraulic graph.
    for label, reservoirs in [("intake",set(feeders)),("upstream",set(upstream))]:
        river_names = {name for r in reservoirs for kind,name in FORWARD.get(("reservoir",r),[]) if kind=="river"}
        requirement = sum((base[f"min_flow__{r}"].to_numpy().reshape(-1,HOURS)
                           for r in river_names if f"min_flow__{r}" in base),np.zeros_like(price))
        flow = state(base,sorted(reservoirs),"inflow").sum(2)
        extra[f"topo_{label}_minflow_requirement"] = requirement
        extra[f"topo_{label}_natural_inflow_minus_requirement"] = flow-requirement
    return extra
def efficiency(spec, head, power):
    head=np.asarray(head,dtype=np.float64)
    curves=sorted(spec["turb_eff_curves"],key=lambda c:c["ref"])
    refs=np.array([c["ref"] for c in curves])
    q=power*1e6/(1000*9.81*head*0.8)
    generator_curve=spec["gen_eff_curve"]
    eta_gen=np.interp(power,generator_curve["x"],np.array(generator_curve["y"])/100)
    for _ in range(4):
        etas=np.stack([np.interp(q,c["x"],np.array(c["y"])/100) for c in curves],axis=-1)
        upper=np.clip(np.searchsorted(refs,head,side="right"),1,max(1,len(refs)-1))
        if len(refs)==1:
            eta_turb=etas[...,0]
        else:
            lower=upper-1
            fraction=np.clip((head-refs[lower])/(refs[upper]-refs[lower]),0,1)
            eta_turb=np.take_along_axis(etas,lower[...,None],axis=-1)[...,0]*(1-fraction)+np.take_along_axis(etas,upper[...,None],axis=-1)[...,0]*fraction
        eta=np.clip(eta_gen*eta_turb,0.3,1.0)
        q=power*1e6/(1000*9.81*head*eta)
    return q,eta

## Unit LP, projected-head sensitivity and actual tree features

In [ ]:
def check_deadline():
    if DEADLINE_UTC and datetime.now(timezone.utc) >= datetime.fromisoformat(DEADLINE_UTC):
        raise TimeoutError("Authorized fitting/preprocessing deadline reached.")


LP_STEP = 6
LP_CACHE = {}
LP_STRUCTURE = None
PLANTS = list(PLANT_GENERATORS)


def conversion_plants(node, seen=None):
    """Follow supplied generating routes; river-only choices use main_river."""
    seen = set() if seen is None else seen
    if node in seen:
        raise ValueError("Cycle in water-value conversion route.")
    if node[0] == "plant":
        return {node[1]}
    outgoing = [(e["to_type"], e["to"]) for e in connections
                if (e["from_type"], e["from"]) == node and e["to_type"] in HYDRAULIC_KINDS]
    if node[0] == "reservoir":
        direct = [t for t in outgoing if t[0] == "plant"]
        tunnels = [t for t in outgoing if t[0] == "tunnel"]
        rivers = [t for t in outgoing if t[0] == "river"]
        if direct or tunnels:
            outgoing = direct or tunnels
        elif rivers:
            main = [t for t in rivers if int(topology["model"]["river"][t[1]].get("main_river", 0)) == 1]
            if len(main) > 1:
                raise ValueError("Ambiguous main-river conversion route.")
            outgoing = main or rivers[:1]
    return set().union(*(conversion_plants(t, seen | {node}) for t in outgoing)) if outgoing else set()


def make_lp_structure():
    """Split plant-outlet flow into parallel, individually bounded unit arcs."""
    from scipy.sparse import lil_matrix
    nodes = sorted(set(FORWARD) | {n for targets in FORWARD.values() for n in targets})
    raw_arcs = sorted((a, b) for a, targets in FORWARD.items() for b in targets)
    raw_arcs += [(n, ("sink", "outside")) for n in nodes if not FORWARD.get(n)]
    arcs = []
    for left, right in raw_arcs:
        if left[0] == "plant":
            outgoing = [b for a, b in raw_arcs if a == left]
            if len(outgoing) != 1:
                raise ValueError("Individual unit arcs require one outlet per plant.")
            arcs.extend((left, right, g) for g in PLANT_GENERATORS[left[1]])
        else:
            arcs.append((left, right, None))
    steps, nr, nf, nn = HOURS // LP_STEP, len(RESERVOIRS), len(arcs), len(nodes)
    positions = {n: i for i, n in enumerate(nodes)}
    nv = steps * (nf + 3 * nr)
    eq = lil_matrix((steps * nn, nv), dtype=float)
    ub = lil_matrix((2 * steps * nr, nv), dtype=float)
    for k in range(steps):
        for j, (a, b, g) in enumerate(arcs):
            eq[k * nn + positions[a], k * nf + j] += 1
            if b in positions:
                eq[k * nn + positions[b], k * nf + j] -= 1
        for j, r in enumerate(RESERVOIRS):
            column = steps * nf + k * nr + j
            row = k * nn + positions[("reservoir", r)]
            eq[row, column] += 1
            if k:
                eq[row, column - nr] -= 1
            upper, lower = 2 * (k * nr + j), 2 * (k * nr + j) + 1
            ub[upper, column], ub[lower, column] = 1, -1
            ub[upper, steps * (nf + nr) + k * nr + j] = -1
            ub[lower, steps * (nf + 2 * nr) + k * nr + j] = -1
    unit_arc = {g: j for j, (_, _, g) in enumerate(arcs) if g is not None}
    if set(unit_arc) != set(GENERATORS):
        raise ValueError("LP must represent all 14 units.")
    return dict(steps=steps, nr=nr, nf=nf, nn=nn, nv=nv, nodes=nodes,
                positions=positions, arcs=arcs, eq=eq.tocsr(), ub=ub.tocsr(), unit_arc=unit_arc)


def heads_from_volume(volume):
    """Volume may be initial or LP projected; it is never future observed state."""
    levels = {r: np.interp(volume[..., j], *LEVEL_CURVES[r]) for j, r in enumerate(RESERVOIRS)}
    heads = {}
    for p in PLANTS:
        intake = np.average(np.stack([levels[r] for r in PLANT_FEEDERS[p]], axis=-1),
                            weights=[CAPACITY[r] for r in PLANT_FEEDERS[p]], axis=-1)
        tail = np.full_like(intake, float(topology["model"]["plant"][p]["outlet_line"]))
        for r in RECEIVERS[p]:
            tail = np.maximum(tail, levels[r])
        heads[p] = np.maximum(intake - tail, 1.)
    return heads


def terminal_conversion(volume, water_values):
    """Approximate local EUR/MWh values as cumulative downstream EUR/Mm3."""
    initial_heads = heads_from_volume(volume)
    factors = {}
    for p in PLANTS:
        grid = [np.linspace(float(GEN_SPECS[g]["p_min"]), float(GEN_SPECS[g]["p_max"]), 33)
                for g in PLANT_GENERATORS[p]]
        q = [efficiency(GEN_SPECS[g], np.full(33, initial_heads[p]), power)[0]
             for g, power in zip(PLANT_GENERATORS[p], grid)]
        factors[p] = float(np.max(sum(grid) * 1e6 / (sum(q) * 3600)))
    selected = {}
    for r in RESERVOIRS:
        candidates = conversion_plants(("reservoir", r))
        if not candidates:
            raise ValueError(f"No terminal energy conversion route for {r}")
        selected[r] = max(candidates, key=lambda p: (factors[p], p))
    local = dict(zip(RESERVOIRS, water_values))
    money = {}
    def recurse(r, seen=None):
        if r in money:
            return money[r]
        seen = set() if seen is None else seen
        if r in seen:
            raise ValueError("Cycle in downstream terminal valuation.")
        p = selected[r]
        receiving = max([0.] + [recurse(t, seen | {r}) for t in RECEIVERS[p]])
        money[r] = float(local[r]) * factors[p] + receiving
        return money[r]
    return np.array([recurse(r) for r in RESERVOIRS]), factors


def solve_unit_lp(start, projected=None, value_multiplier=1.):
    """Continuous relaxation with mass balance and soft storage limits, not SHOP."""
    from scipy.optimize import linprog
    import warnings
    check_deadline()
    global LP_STRUCTURE
    if LP_STRUCTURE is None:
        LP_STRUCTURE = make_lp_structure()
    s = LP_STRUCTURE
    k, nr, nf, nn = [s[x] for x in ("steps", "nr", "nf", "nn")]
    times = pd.date_range(start, periods=HOURS, freq="h")
    initial = lookup("volumes", [start], RESERVOIRS)[0].astype(float)
    values = lookup("water_values", [start + pd.Timedelta(hours=HOURS)], RESERVOIRS)[0].astype(float)
    price = lookup("prices", times).reshape(k, LP_STEP).mean(1)
    inflows = tables["inflows"].reindex(times)
    minflows = tables["min_flow"].reindex(times)
    minvolumes = tables["min_volume"].reindex(times)
    block_volumes = np.tile(initial, (k, 1)) if projected is None else np.vstack([initial, projected[:-1]])
    heads = heads_from_volume(block_volumes)
    qmax, factor = {}, {}
    for g in GENERATORS:
        power = float(GEN_SPECS[g]["p_max"])
        qmax[g] = efficiency(GEN_SPECS[g], heads[PLANT_FOR[g]], power)[0]
        factor[g] = power * 1e6 / (qmax[g] * 3600)
    money, best_factor = terminal_conversion(initial, values * value_multiplier)
    objective = np.zeros(s["nv"])
    rhs = np.zeros(k * nn)
    limits = np.zeros(2 * k * nr)
    bounds = []
    seconds = LP_STEP * 3600
    for step in range(k):
        for j, (left, right, g) in enumerate(s["arcs"]):
            low, high = 0., 1000. * seconds / 1e6
            if g is not None:
                high = qmax[g][step] * seconds / 1e6
                objective[step * nf + j] = -price[step] * factor[g][step]
            if left[0] == "river" and left[1] in minflows:
                low = float(minflows[left[1]].to_numpy().reshape(k, LP_STEP)[step].sum()) * 3600 / 1e6
            bounds.append((low, max(high, low)))
        for node, position in s["positions"].items():
            if node[0] in {"reservoir", "river"} and node[1] in inflows:
                rhs[step * nn + position] = float(inflows[node[1]].to_numpy().reshape(k, LP_STEP)[step].sum()) * 3600 / 1e6
        for j, r in enumerate(RESERVOIRS):
            if step == 0:
                rhs[s["positions"][("reservoir", r)]] += initial[j]
            limits[2 * (step * nr + j)] = CAPACITY[r]
            floor = float(minvolumes[r].to_numpy().reshape(k, LP_STEP)[step].max()) if r in minvolumes else 0.
            limits[2 * (step * nr + j) + 1] = -floor
    bounds.extend([(0, None)] * (3 * k * nr))
    objective[k * nf + (k - 1) * nr:k * (nf + nr)] = -money
    objective[k * (nf + nr):] = 1e6
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="Unrecognized options detected")
        fit = linprog(objective, A_eq=s["eq"], b_eq=rhs, A_ub=s["ub"], b_ub=limits,
                      bounds=bounds, method="highs", options={"threads": 1, "time_limit": 20.})
    if not fit.success:
        raise RuntimeError(f"Unit LP failed at {start}: {fit.message}")
    residual = float(np.max(np.abs(s["eq"] @ fit.x - rhs)))
    if residual > 1e-5:
        raise ValueError("LP water-balance residual exceeds tolerance.")
    flows = fit.x[:k * nf].reshape(k, nf)
    fractions = np.column_stack([flows[:, s["unit_arc"][g]] / (qmax[g] * seconds / 1e6) for g in GENERATORS])
    storage = fit.x[k * nf:k * (nf + nr)].reshape(k, nr)
    slack = fit.x[k * (nf + nr):].reshape(2, k, nr).sum(axis=(0, 2))
    shadows = fit.eqlin.marginals.reshape(k, nn)
    margins = np.column_stack([(price * factor[g] + shadows[:, s["positions"][("plant", PLANT_FOR[g])]])
                               / factor[g] for g in GENERATORS])
    return dict(fractions=fractions, storage=storage, margins=margins, slack=slack,
                heads=np.column_stack([heads[p] for p in PLANTS]), residual=residual,
                objective=float(fit.fun), terminal_money=money)


def lp_case(start):
    start = pd.Timestamp(start)
    if start not in LP_CACHE:
        base = solve_unit_lp(start)
        corrected = solve_unit_lp(start, projected=base["storage"])
        low = solve_unit_lp(start, projected=base["storage"], value_multiplier=.95)
        high = solve_unit_lp(start, projected=base["storage"], value_multiplier=1.05)
        LP_CACHE[start] = dict(base=base, corrected=corrected, low=low, high=high)
    return LP_CACHE[start]


def precompute_lp(starts):
    """Case-local read-only inputs; four solvers with one thread each."""
    from concurrent.futures import ThreadPoolExecutor
    global LP_STRUCTURE
    if LP_STRUCTURE is None:
        LP_STRUCTURE = make_lp_structure()
    missing = [s for s in pd.DatetimeIndex(starts).unique() if s not in LP_CACHE]
    with ThreadPoolExecutor(max_workers=4) as pool:
        for i, _ in enumerate(pool.map(lp_case, missing)):
            if (i + 1) % 100 == 0:
                print(f"LP cases {i + 1}/{len(missing)}", flush=True)


def relative_features(base):
    """One shared case scale preserves signed relative margins across regimes."""
    price = base["price"].to_numpy().reshape(-1, HOURS)
    local = {p: plant_state(base, p)["value"] for p in PLANTS}
    values = np.stack([local[p][:, 0] for p in PLANTS], axis=1)
    scale = np.maximum(np.median(np.abs(np.concatenate([price, values], axis=1)), axis=1), 1.)[:, None]
    result = {}
    # Nine local opportunity values plus the eight available intake/receiver differences.
    # Explicit route names, rather than arbitrary row positions, define the pairs.
    for p in PLANTS:
        margin = (price - local[p]) / scale
        result[f"relative_local__{p}"] = margin
        result[f"relative_local_week_mean__{p}"] = np.repeat(margin.mean(1)[:, None], HOURS, axis=1)
    pairs = [p for p in PLANTS if RECEIVERS[p]]
    for p in pairs:
        receiver = RECEIVERS[p]
        receiving = np.average(state(base, receiver, "terminal_value"),
                               weights=[CAPACITY[r] for r in receiver], axis=2)
        # Water values are local EUR/MWh, so this is a relative-value descriptor,
        # not the exact profit or water cost of a cascade transfer.
        margin = (price - (local[p] - receiving)) / scale
        result[f"relative_net__{p}"] = margin
        result[f"relative_net_week_mean__{p}"] = np.repeat(margin.mean(1)[:, None], HOURS, axis=1)
    return result


def nine_summaries(values, prefix):
    """Within-case summaries; no recurrence crosses the 168-hour boundary."""
    positive = values > 1e-7
    elapsed = np.zeros_like(values)
    remaining = np.zeros_like(values)
    for h in range(HOURS):
        elapsed[:, h] = np.where(positive[:, h], (elapsed[:, h - 1] if h else 0) + 1, 0)
    for h in range(HOURS - 1, -1, -1):
        remaining[:, h] = np.where(positive[:, h], (remaining[:, h + 1] if h < HOURS - 1 else 0) + 1, 0)
    return {f"{prefix}_{key}": value for key, value in {
        "fraction": values, "on_proxy": positive.astype(float),
        "week_mean": np.repeat(values.mean(1)[:, None], HOURS, axis=1),
        "remaining_mean": values[:, ::-1].cumsum(1)[:, ::-1] / np.arange(HOURS, 0, -1),
        "elapsed_mean": values.cumsum(1) / np.arange(1, HOURS + 1),
        "remaining_on_hours": positive[:, ::-1].cumsum(1)[:, ::-1],
        "elapsed_on_hours": positive.cumsum(1), "positive_streak_elapsed": elapsed,
        "positive_streak_remaining": remaining}.items()}


def hydraulic_features(base, generator, corrected=False):
    data = [lp_case(s) for s in base.attrs["case_starts"]]
    j = GENERATORS.index(generator)
    p = PLANT_FOR[generator]
    repeat = lambda x: np.repeat(x, LP_STEP, axis=1)
    fractions = np.stack([x["base"]["fractions"] for x in data])
    margins = np.stack([x["base"]["margins"] for x in data])
    result = {}
    for plant in PLANTS:
        indices = [GENERATORS.index(g) for g in PLANT_GENERATORS[plant]]
        powers = np.array([float(GEN_SPECS[g]["p_max"]) for g in PLANT_GENERATORS[plant]])
        result[f"lp_plant_fraction__{plant}"] = repeat(np.average(fractions[:, :, indices], weights=powers, axis=2))
        result[f"lp_plant_margin__{plant}"] = repeat(np.average(margins[:, :, indices], weights=powers, axis=2))
    result.update(nine_summaries(repeat(fractions[:, :, j]), "lp_own"))
    if corrected:
        scenarios = {key: repeat(np.stack([x[key]["fractions"][:, j] for x in data]))
                     for key in ("corrected", "low", "high")}
        result.update(nine_summaries(scenarios["corrected"], "lp_head_own"))
        for key in ("low", "high"):
            result[f"lp_{key}_value_fraction"] = scenarios[key]
            result[f"lp_{key}_value_on_proxy"] = (scenarios[key] > 1e-7).astype(float)
            result[f"lp_{key}_minus_head_fraction"] = scenarios[key] - scenarios["corrected"]
        result["lp_head_minus_initial_fraction"] = scenarios["corrected"] - repeat(fractions[:, :, j])
        result["lp_head_gross_head_m"] = repeat(np.stack([x["corrected"]["heads"][:, PLANTS.index(p)] for x in data]))
        storage = np.stack([x["corrected"]["storage"] for x in data])
        for label, reservoirs in [("intake", FEEDERS[generator]), ("receiver", RECEIVERS[p])]:
            indices = [RESERVOIRS.index(r) for r in reservoirs]
            value = storage[:, :, indices].sum(2) / sum(CAPACITY[r] for r in reservoirs) if indices else np.zeros(storage.shape[:2])
            result[f"lp_head_projected_{label}_fraction"] = repeat(value)
        result["lp_head_soft_violation_Mm3"] = repeat(np.stack([x["corrected"]["slack"] for x in data]))
        result["lp_sensitivity_on_fraction"] = np.mean(np.stack([v > 1e-7 for v in scenarios.values()]), axis=0)
        if len(PLANT_GENERATORS[p]) > 1:
            indices = [GENERATORS.index(g) for g in PLANT_GENERATORS[p]]
            powers = [float(GEN_SPECS[g]["p_max"]) for g in PLANT_GENERATORS[p]]
            plant_schedule = repeat(np.stack([np.average(x["corrected"]["fractions"][:, indices], weights=powers, axis=1) for x in data]))
            result.update(nine_summaries(plant_schedule, "lp_own_plant_on"))
    return result


def model_features(base, generator, variant):
    if variant not in VARIANTS:
        raise ValueError(f"Unknown feature variant: {variant}")
    frame = reference_generator_features(base, generator)
    extra = topology_features(base, generator)
    if variant != "base_topology":
        extra.update(relative_features(base))
    if variant in {"unit_lp", "head_sensitivity", "head_no_lio_relative"}:
        extra.update(hydraulic_features(base, generator, corrected=variant != "unit_lp"))
    if variant == "head_no_lio_relative" and generator == "Lio_G1":
        extra = {k: v for k, v in extra.items() if not k.startswith("relative_")}
    result = pd.concat([frame, pd.DataFrame({k: np.asarray(v, dtype=np.float32).reshape(-1)
                                           for k, v in extra.items()}, index=base.index)], axis=1)
    if not result.columns.is_unique or not np.isfinite(result.to_numpy()).all():
        raise ValueError("Model features require unique names and finite values.")
    return result.astype(np.float32, copy=False)


## Purged partitions, fitting, metric and availability checks

In [ ]:
def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False, default=str), encoding="utf-8")
    temporary.replace(path)


def partition(year):
    start = pd.Timestamp(f"{year}-01-01", tz="UTC")
    end = pd.Timestamp(f"{year + 1}-01-01", tz="UTC")
    ends = train_cases.starttime + pd.Timedelta(hours=HOURS)
    fitting = train_cases.loc[ends <= start].copy()
    scoring = train_cases.loc[(train_cases.starttime >= start) & (ends <= end)].copy()
    assert len(fitting) and len(scoring)
    assert (fitting.starttime + pd.Timedelta(hours=HOURS)).max() <= scoring.starttime.min()
    return fitting, scoring


def assemble(cases, predictions):
    if set(predictions) != set(GENERATORS):
        raise ValueError("A complete result requires all 14 unit predictions.")
    output = np.empty((len(cases), len(LABELS)), dtype=np.float32)
    for g in GENERATORS:
        value = np.asarray(predictions[g])
        if value.shape != (len(cases), HOURS):
            raise ValueError(f"Wrong prediction shape for {g}")
        output[:, TARGET_POSITIONS[g]] = value
    if not np.isfinite(output).all() or np.any((output < 0) | (output > 1)):
        raise ValueError("Output must contain finite probabilities in [0, 1].")
    frame = pd.DataFrame(output, columns=LABELS)
    frame.insert(0, "Run No", cases["Run No"].to_numpy())
    return frame


def official_score(cases, predictions):
    """Inline micro AUC; audited against the supplied course metric on native outputs."""
    from sklearn.metrics import roc_auc_score
    predicted = assemble(cases, predictions)
    actual = cases.set_index("Run No")[LABELS]
    aligned = predicted.set_index("Run No").loc[actual.index, LABELS]
    return float(roc_auc_score(actual.to_numpy(), aligned.to_numpy(), average="micro"))



def dependency_versions():
    return {name: metadata.version(name) for name in
            ["lightgbm", "numpy", "pandas", "scipy", "scikit-learn", "PyYAML", "psutil", "nbformat"]}


def fit_units(fitting, scoring, variant, folder, *, rounds=None, units=None, max_rounds=MAX_ROUNDS, seed=SEED):
    """Fit disjoint unit targets; BCE stopping is development selection, not a holdout."""
    import lightgbm as lgb
    from sklearn.metrics import roc_auc_score, log_loss
    import psutil
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    units = GENERATORS if units is None else units
    check_deadline()
    started = perf_counter()
    # Inputs receive only metadata; held-out labels cannot enter feature functions.
    fit_base = feature_matrix(fitting[["Run No", "starttime"]])
    score_base = feature_matrix(scoring[["Run No", "starttime"]])
    predictions, records = {}, []
    for g in units:
        check_deadline()
        path = folder / (g.replace(" ", "_") + ".json")
        if path.exists():
            raise FileExistsError(f"Refusing to overwrite an existing fit: {path}")
        tick = perf_counter()
        x = model_features(fit_base, g, variant)
        z = model_features(score_base, g, variant)
        if list(x.columns) != list(z.columns):
            raise ValueError("Feature schema differs between fitting and scoring.")
        names = list(x.columns)
        write_json(folder / (g.replace(" ", "_") + "_features.json"), names)
        # Safe numeric feature IDs avoid LightGBM JSON-name restrictions; save the exact mapping.
        matrix = lgb.Dataset(x.to_numpy(copy=False), label=labels_for(fitting, g),
                             feature_name=[f"f{i:04d}" for i in range(len(names))], free_raw_data=True)
        evaluation = lgb.Dataset(z.to_numpy(copy=False), label=labels_for(scoring, g),
                                 reference=matrix, free_raw_data=True) if rounds is None else None
        def deadline_callback(env):
            check_deadline()
        deadline_callback.order = 5
        history = {}
        callbacks = [deadline_callback, lgb.record_evaluation(history)]
        if rounds is None:
            callbacks.append(lgb.early_stopping(STOPPING_ROUNDS, first_metric_only=True, verbose=False))
        parameters = {**PARAMETERS, "seed": seed, "bagging_seed": seed,
                      "feature_fraction_seed": seed, "data_random_seed": seed}
        fitted = lgb.train(parameters, matrix, num_boost_round=max_rounds if rounds is None else int(rounds[g]),
                           valid_sets=[evaluation] if evaluation is not None else None,
                           valid_names=["development"] if evaluation is not None else None, callbacks=callbacks)
        best = int(fitted.best_iteration or fitted.current_iteration())
        probability = fitted.predict(z.to_numpy(copy=False), num_iteration=best, num_threads=THREADS).astype(np.float32)
        fitted.save_model(str(folder / (g.replace(" ", "_") + ".txt")), num_iteration=best)
        predictions[g] = probability.reshape(len(scoring), HOURS)
        np.save(folder / (g.replace(" ", "_") + "_probability.npy"), predictions[g])
        importance = pd.DataFrame({"feature": names, "gain": fitted.feature_importance("gain", iteration=best)})
        importance.to_csv(folder / (g.replace(" ", "_") + "_importance.csv"), index=False)
        y = labels_for(scoring, g) if set(LABELS).issubset(scoring.columns) else None
        record = dict(generator=g, variant=variant, fitting_cases=len(fitting), scoring_cases=len(scoring),
                      features=len(names), best_iteration=best, trained_iterations=len(history.get("development", {}).get("binary_logloss", [])) or fitted.current_iteration(),
                      seed=seed, auc=float(roc_auc_score(y, probability)) if y is not None and np.unique(y).size == 2 else None,
                      logloss=float(log_loss(y, probability, labels=[0, 1])) if y is not None else None,
                      seconds=perf_counter() - tick, process_rss_GiB=psutil.Process().memory_info().rss / 2**30,
                      parameters=parameters, stopping_uses_scoring_labels=rounds is None)
        records.append(record)
        write_json(path, record)
        write_json(folder / (g.replace(" ", "_") + "_curve.json"), history)
        print(f"{variant} {scoring.starttime.iloc[0].year} {g}: trees={best} features={len(names)} AUC={record['auc']} seconds={record['seconds']:.1f}", flush=True)
        del fitted, matrix, evaluation, x, z
        gc.collect()
    summary = dict(variant=variant, fitting_ids=fitting["Run No"].tolist(), scoring_ids=scoring["Run No"].tolist(),
                   seed=seed, units=records, seconds=perf_counter() - started,
                   complete=set(predictions) == set(GENERATORS))
    if summary["complete"]:
        summary["micro_auc"] = official_score(scoring, predictions) if set(LABELS).issubset(scoring.columns) else None
        np.save(folder / "prediction_matrix.npy", assemble(scoring, predictions)[LABELS].to_numpy())
    write_json(folder / "summary.json", summary)
    return summary, predictions


def audit_features(folder):
    """Check isolation, availability, schema and conservative LP before fitting."""
    folder = Path(folder)
    cases = pd.concat([train_cases.iloc[[365, 1826, 2557]], test_cases.iloc[[0, -1]]], ignore_index=True)
    precompute_lp(cases.starttime)
    base = feature_matrix(cases[["Run No", "starttime"]])
    inventory = []
    for g in GENERATORS:
        for variant in VARIANTS:
            x = model_features(base, g, variant)
            inventory.extend(dict(generator=g, variant=variant, feature=name,
                                  group="relative" if name.startswith("relative_") else "lp" if name.startswith("lp_") else "base_topology")
                             for name in x.columns)
    pd.DataFrame(inventory).to_csv(folder / "feature_inventory.csv", index=False)
    chosen = cases.iloc[[1]].copy()
    before = model_features(feature_matrix(chosen), "Lio_G1", "head_sensitivity")
    together = model_features(base, "Lio_G1", "head_sensitivity").iloc[HOURS:2 * HOURS]
    np.testing.assert_array_equal(before.to_numpy(), together.to_numpy())
    old = tables["volumes"].copy()
    preserved_cache = LP_CACHE.copy()
    try:
        tables["volumes"].loc[tables["volumes"].index > chosen.starttime.iloc[0]] *= .2
        LP_CACHE.clear()
        after = model_features(feature_matrix(chosen), "Lio_G1", "head_sensitivity")
        np.testing.assert_array_equal(before.to_numpy(), after.to_numpy())
    finally:
        tables["volumes"] = old
        LP_CACHE.clear()
        LP_CACHE.update(preserved_cache)
    changed = chosen.copy()
    changed[LABELS] = 1 - changed[LABELS]
    np.testing.assert_array_equal(before.to_numpy(), model_features(feature_matrix(changed), "Lio_G1", "head_sensitivity").to_numpy())
    for value in LP_CACHE.values():
        for lp in value.values():
            assert lp["residual"] < 1e-5
            assert np.min(lp["storage"]) >= -1e-7
            assert np.min(lp["fractions"]) >= -1e-7 and np.max(lp["fractions"]) <= 1 + 1e-7
    first = model_features(base, "Lio_G1", "head_sensitivity")
    ablated = model_features(base, "Lio_G1", "head_no_lio_relative")
    assert list(ablated.columns) == [c for c in first if not c.startswith("relative_")]
    for year in [2020, 2021, 2022]:
        partition(year)
    fake = {g: np.full((len(cases), HOURS), (i + 1) / 15, dtype=np.float32) for i, g in enumerate(GENERATORS)}
    assembled = assemble(cases, fake)
    for g in GENERATORS:
        np.testing.assert_array_equal(assembled[TARGET_COLUMNS[g]].to_numpy(), fake[g])
    result = dict(status="passed", cases=len(cases), future_storage_invariant=True,
                  label_invariant=True, case_isolation=True, mapping=True,
                  max_lp_residual=max(x["residual"] for v in LP_CACHE.values() for x in v.values()),
                  minflow_and_storage_are_block_approximations=True, classifiers_fitted=False,
                  counts=pd.DataFrame(inventory).groupby(["generator", "variant"]).size().unstack().to_dict(),
                  versions=dependency_versions())
    write_json(folder / "feature_checks.json", result)
    return result

In [ ]:
def implementation_identity():
    """Hash executed function bytecode/constants without filename or line numbers."""
    from types import CodeType
    def encode(value):
        if isinstance(value, CodeType):
            return dict(bytecode=value.co_code.hex(), names=value.co_names,
                        variables=value.co_varnames, freevars=value.co_freevars,
                        cellvars=value.co_cellvars, argcount=value.co_argcount,
                        posonly=value.co_posonlyargcount, kwonly=value.co_kwonlyargcount,
                        flags=value.co_flags, exceptiontable=value.co_exceptiontable.hex(),
                        constants=[encode(v) for v in value.co_consts])
        if isinstance(value, (tuple, list)):
            return [encode(v) for v in value]
        if isinstance(value, (set, frozenset)):
            return sorted([encode(v) for v in value], key=repr)
        if isinstance(value, bytes):
            return {"bytes": value.hex()}
        if value is None or isinstance(value, (str, int, float, bool)):
            return value
        return {"literal": repr(value)}
    names = ["economic_context", "water_context", "lookup", "feature_matrix", "reference_feature_matrix",
             "reference_generator_features", "state", "plant_state", "topology_features", "efficiency",
             "conversion_plants", "make_lp_structure", "heads_from_volume", "terminal_conversion",
             "solve_unit_lp", "lp_case", "precompute_lp", "relative_features", "nine_summaries",
             "hydraulic_features", "model_features", "partition", "assemble", "fit_units"]
    payload = {name: encode(globals()[name].__code__) for name in names}
    payload["global_settings"] = dict(hours=HOURS, lp_step=LP_STEP, threads=THREADS, seed=SEED,
                                     decision_groups=sorted(DECISION_GROUPS), feature_version=FEATURE_VERSION,
                                     parameters=PARAMETERS, max_rounds=MAX_ROUNDS, stopping_rounds=STOPPING_ROUNDS)
    payload["target_schema"] = dict(generators=GENERATORS, reservoirs=RESERVOIRS,
        columns=TARGET_COLUMNS, positions=TARGET_POSITIONS)
    payload["named_graph"] = dict(plants=PLANTS, plant_units=PLANT_GENERATORS,
        unit_plants=PLANT_FOR, feeders=FEEDERS, upstream=UPSTREAM, receivers=RECEIVERS,
        upstream_plants=UPSTREAM_PLANTS, downstream_plants=DOWNSTREAM_PLANTS,
        forward=sorted((list(a), sorted(list(b) for b in targets)) for a, targets in FORWARD.items()),
        capacities=CAPACITY, levels={r: [x.tolist(), y.tolist()] for r, (x, y) in LEVEL_CURVES.items()},
        topology_sha256=hashlib.sha256(yaml.safe_dump(topology, sort_keys=True).encode()).hexdigest())
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()


def guard_selected(configuration):
    if not isinstance(configuration, dict):
        raise ValueError("A measured frozen LightGBM configuration is required.")
    c = configuration
    expected = globals().get("FROZEN_CONFIGURATION_SHA256")
    actual = hashlib.sha256(json.dumps(c, sort_keys=True).encode()).hexdigest()
    if expected is None or actual != expected:
        raise ValueError("Frozen configuration checksum differs from the measured recipe.")
    if c["parameters"] != PARAMETERS or c["versions"] != dependency_versions() or c["seed"] != SEED:
        raise ValueError("Frozen parameters, versions or seed differ from the measured recipe.")
    if c["implementation_identity"] != implementation_identity():
        raise ValueError("Executed implementation differs from the measured recipe.")
    if c["variant"] not in VARIANTS or set(c["iterations"]) != set(GENERATORS):
        raise ValueError("Frozen variant/tree budget is incomplete.")
    if any(type(v) is not int or v < 1 for v in c["iterations"].values()):
        raise ValueError("Frozen tree counts must be positive integers.")
    for key, path in RAW_FILES.items():
        if hashlib.sha256(path.read_bytes()).hexdigest() != c["raw_sha256"][key]:
            raise ValueError(f"Raw resource differs from the measured data: {key}")
    probe = feature_matrix(train_cases.iloc[[0]][["Run No", "starttime"]])
    for g in GENERATORS:
        names = list(model_features(probe, g, c["variant"]).columns)
        signature = hashlib.sha256(json.dumps(names).encode()).hexdigest()
        if signature != c["feature_schema_sha256"][g]:
            raise ValueError(f"Frozen feature inventory differs for {g}")
    return c


In [ ]:
if RUN_MODE == "check":
    folder = OUTPUT_ROOT / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_check")
    folder.mkdir(parents=True, exist_ok=False)
    audit_features(folder)
    guard_selected(SELECTED_CONFIGURATION)
    print(f"Non-fitting checks passed: {folder}")
elif RUN_MODE in {"validate", "holdout", "submit"}:
    c = guard_selected(SELECTED_CONFIGURATION)
    folder = OUTPUT_ROOT / (datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + RUN_MODE)
    folder.mkdir(parents=True, exist_ok=False)
    if RUN_MODE == "validate":
        results = []
        for year in [2020, 2021]:
            fitting, scoring = partition(year)
            if c["variant"] not in {"base_topology", "relative"}:
                precompute_lp(pd.concat([fitting.starttime, scoring.starttime]))
            summary, _ = fit_units(fitting, scoring, c["variant"], folder / str(year))
            results.append(summary)
        write_json(folder / "validation.json", results)
        print("Equal-year development micro AUC:", np.mean([x["micro_auc"] for x in results]))
    elif RUN_MODE == "holdout":
        fitting, scoring = partition(2022)
        if c["variant"] not in {"base_topology", "relative"}:
            precompute_lp(pd.concat([fitting.starttime, scoring.starttime]))
        summary, _ = fit_units(fitting, scoring, c["variant"], folder, rounds=c["iterations"])
        print("Exposed-2022 diagnostic micro AUC:", summary["micro_auc"])
    else:
        if c["variant"] not in {"base_topology", "relative"}:
            precompute_lp(pd.concat([train_cases.starttime, test_cases.starttime]))
        summary, predictions = fit_units(train_cases, test_cases, c["variant"], folder,
                                         rounds=c["iterations"], seed=c["seed"])
        frame = assemble(test_cases, predictions)
        assert list(frame.columns) == list(template.columns)
        np.testing.assert_array_equal(frame["Run No"], template["Run No"])
        frame.to_csv(folder / "lightgbm_submission.csv", index=False)
        print(folder / "lightgbm_submission.csv")
else:
    raise ValueError("Use check, validate, holdout or submit.")
